# ML_U4_Lab01 — K-means en Práctica

**Versión:** 2025-1 | **Modificado:** 2026-05-30
**Dataset:** Iris + make_blobs | **Duración:** 1 hora
**Modalidad:** Individual o parejas

---

## 📋 Estructura del laboratorio

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|-----------|
| Setup | Imports y datos | 5 min | Todos |
| PARTE 1 | Sin computador: razonamiento sobre K-means | 10 min | Todos |
| PARTE 2 | Implementación K-means desde cero | 20 min | Todos |
| PARTE 3 | Selección de K y aplicación a Iris | 20 min | Todos |
| ANÁLISIS | Interpretación y reflexión | 5 min | Diferenciado |

---

## 🎯 Instrucciones por audiencia

| | Pregrado | Doctorado |
|--|----------|----------|
| Obligatorio | Partes 1, 2, 3 + preguntas azules | Todo lo anterior + TODOs [PhD] + preguntas amarillas |
| Opcional | Bonus azul | Bonus amarillo |
| Entrega | .ipynb ejecutado | .ipynb ejecutado |

## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Datasets
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_blobs, y_blobs = make_blobs(n_samples=300, centers=[
    [-3, -3], [-3, 3], [3, -3], [3, 3]
], cluster_std=0.9, random_state=RANDOM_STATE)

import sklearn
print(f"✅ Setup completo | sklearn {sklearn.__version__}")
print(f"   Iris: {X_iris.shape} | Blobs: {X_blobs.shape}")

---
## PARTE 1 — Sin Computador: Razonamiento sobre K-means (10 min) 🖊️

In [ ]:
# ━━━ PARTE 1: VERIFICACIÓN ━━━
data_1d = np.array([2., 4., 10., 12., 20., 22.]).reshape(-1, 1)
km_verify = KMeans(n_clusters=3, init=np.array([[2.], [10.], [20.]]),
                   n_init=1, random_state=RANDOM_STATE)
labels_verify = km_verify.fit_predict(data_1d)

print("Verificación manual:")
for x, l in zip(data_1d.ravel(), labels_verify):
    print(f"  x={x:.0f} → Cluster {l}")
print(f"\nCentroides finales: {km_verify.cluster_centers_.ravel()}")
print(f"Inercia final: {km_verify.inertia_:.2f}")

---
## PARTE 2 — K-means desde Cero (20 min)

In [ ]:
# ━━━ TODO 1: IMPLEMENTAR K-MEANS DESDE CERO ━━━
# Implementa el algoritmo K-means usando solo numpy.
# La función debe:
#   - Inicializar K centroides aleatoriamente (init='random') o con K-means++ (init='kmeans++')
#   - Iterar hasta convergencia (cuando las asignaciones no cambian)
#   - Retornar: (labels, centroids, inertia, n_iter)
#
# Pista para K-means++:
#   1. Elige el primer centroide al azar
#   2. Para cada siguiente centroide: calcula D²(x) = min_k d(x, μ_k)²
#      y muestrea con probabilidad proporcional a D²(x)

def kmeans_numpy(X, K, init='kmeans++', max_iter=300, random_state=None):
    """
    K-means desde cero.
    Retorna: labels (n,), centroids (K, d), inertia (float), n_iter (int)
    """
    rng = np.random.RandomState(random_state)
    n, d = X.shape

    # TODO 1a: Inicialización
    if init == 'random':
        # Elige K puntos al azar como centroides
        # ESCRIBE TU CÓDIGO AQUÍ:
        centers = None  # reemplaza
    elif init == 'kmeans++':
        # K-means++ initialization
        # ESCRIBE TU CÓDIGO AQUÍ:
        centers = None  # reemplaza

    # TODO 1b: Loop de K-means
    # ESCRIBE TU CÓDIGO AQUÍ:
    labels = None
    n_iter = 0
    # (Pista: itera hasta max_iter o hasta que labels no cambie)

    # TODO 1c: Calcular inercia final
    # ESCRIBE TU CÓDIGO AQUÍ:
    inertia = None

    return labels, centers, inertia, n_iter


# Test básico
X_test = StandardScaler().fit_transform(X_blobs)
labels_np, centers_np, inertia_np, n_iter_np = kmeans_numpy(X_test, K=4, random_state=RANDOM_STATE)

if labels_np is not None:
    print(f"K-means numpy: K=4 | Inercia={inertia_np:.2f} | Iteraciones={n_iter_np}")
else:
    print("❌ TODO 1 sin implementar")

In [ ]:
# 🔍 Tests de sanidad — Parte 2 (NO MODIFICAR)
km_ref = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=1, init='k-means++')
km_ref.fit(X_test)

try:
    assert labels_np is not None, "TODO 1: función no implementada"
    assert len(labels_np) == len(X_test), "Labels: longitud incorrecta"
    assert len(np.unique(labels_np)) == 4, "Deben haber exactamente 4 clusters"
    print(f"✅ PASS — Labels shape y clusters OK")
except AssertionError as e:
    print(f"❌ FAIL — {e}")

try:
    assert inertia_np is not None, "Inercia no calculada"
    # Tolerancia del 5% respecto a sklearn (puede diferir por inicialización)
    ratio = inertia_np / km_ref.inertia_
    assert 0.9 <= ratio <= 1.5, f"Inercia {inertia_np:.2f} muy diferente de sklearn {km_ref.inertia_:.2f}"
    print(f"✅ PASS — Inercia OK: {inertia_np:.2f} vs sklearn {km_ref.inertia_:.2f} (ratio={ratio:.3f})")
except AssertionError as e:
    print(f"❌ FAIL — {e}")

---
## PARTE 3 — Selección de K y Aplicación a Iris (20 min)

In [ ]:
# ━━━ TODO 2: SELECCIÓN DE K CON CODO Y SILHOUETTE ━━━
# Aplica K-means a Iris (normalizado) para K ∈ {2, 3, 4, 5, 6, 7}
# Para cada K calcula:
#   - Inercia (km.inertia_)
#   - Silhouette score (silhouette_score)
# Grafica ambas métricas y marca el K óptimo según cada criterio

# ESCRIBE TU CÓDIGO AQUÍ:
X_iris_sc = StandardScaler().fit_transform(X_iris)
K_range = range(2, 8)
inertias = []
silhouettes = []

# for k in K_range:
#     km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
#     labels = km.fit_predict(X_iris_sc)
#     inertias.append(...)      # inercia del modelo
#     silhouettes.append(...)   # silhouette_score(X_iris_sc, labels)

best_k_sil = None  # K con mayor silhouette

if len(silhouettes) > 0:
    print(f"Mejor K por silhouette: {best_k_sil}")

In [ ]:
# ━━━ TODO 3: APLICAR K-MEANS CON EL K ÓPTIMO Y COMPARAR CON VERDAD REAL ━━━
# 1. Entrena KMeans con el K que encontraste (o usa K=3 si no lo encontraste)
# 2. Calcula ARI respecto a y_iris
# 3. Visualiza los clusters en 2D usando PCA (ya creado con n_components=2)
#    - Subgráfico izquierdo: etiquetas reales de Iris
#    - Subgráfico derecho: clusters de K-means
#    - Marca los centroides con una estrella negra
# 4. Imprime una tabla cruzada: pd.crosstab(y_iris, labels_optimo)
#    para ver cómo los clusters se corresponden con las especies

# ESCRIBE TU CÓDIGO AQUÍ:
K_optimo = best_k_sil if best_k_sil is not None else 3

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_iris_2d = pca.fit_transform(X_iris_sc)

# km_optimo = KMeans(...)
# labels_optimo = ...
# ari = adjusted_rand_score(y_iris, labels_optimo)

labels_optimo = None  # reemplaza
ari = None

In [ ]:
# 🔍 Tests de sanidad — Parte 3 (NO MODIFICAR)
try:
    assert len(inertias) == len(list(K_range)), f"Faltan inercias: {len(inertias)} de {len(list(K_range))}"
    assert len(silhouettes) == len(list(K_range)), "Faltan silhouettes"
    assert all(s >= 0 for s in silhouettes), "Silhouette negativo — verifica el cálculo"
    print(f"✅ PASS — Métricas calculadas para K={list(K_range)}")
except (AssertionError, TypeError) as e:
    print(f"❌ FAIL — {e}")

try:
    assert labels_optimo is not None, "TODO 3: labels no calculados"
    assert len(labels_optimo) == len(y_iris), "Labels con longitud incorrecta"
    assert ari is not None and 0.0 <= ari <= 1.0, "ARI fuera de rango"
    print(f"✅ PASS — K-means K={K_optimo} | ARI={ari:.4f}")
except (AssertionError, TypeError) as e:
    print(f"❌ FAIL — {e}")

---
## BONUS (Opcional)

---
## ✅ Checklist de Entrega

### Pregrado
- [ ] Parte 1: preguntas 1–4 respondidas a mano
- [ ] TODO 1: K-means numpy implementado y tests pasando
- [ ] TODO 2: codo + silhouette graficados y K óptimo identificado
- [ ] TODO 3: visualización con PCA + tabla cruzada
- [ ] Preguntas de análisis 1–3 respondidas

### Doctorado (adicional)
- [ ] Parte 1: preguntas 5–7 respondidas
- [ ] TODO [PhD]: K-means con múltiples métricas implementado
- [ ] Preguntas P4–P5 respondidas